# Indic Embedding Language Alignment Benchmark — Colab Version

This notebook benchmarks Indic/multilingual embedding models for **cross-lingual semantic alignment**.

Main question:

> If two sentences mean the same thing in English and an Indic language, do their embeddings become close?

We test this using FLORES translation pairs and compute:

- **Positive cosine mean**: similarity of true translation pairs
- **Random cosine mean**: similarity of mismatched pairs
- **Cosine gap**: positive mean − random mean
- **Accuracy@1**: whether the correct translation is the nearest neighbor
- **Recall@5 / Recall@10 / MRR**

Start small with `QUICK_N = 100`, then increase to 250 or full devtest.


## 0. Colab GPU setup

In Colab, go to:

**Runtime → Change runtime type → Hardware accelerator → GPU → Save**

Then run the next cell.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not enabled. In Colab: Runtime -> Change runtime type -> GPU')

## 1. Install dependencies

Run this once per fresh Colab session. After installing, Colab may ask you to restart the runtime. If it does, restart and continue from the next cells.

In [ ]:
!pip -q install -U   'torch'   'transformers>=4.42.0'   'datasets>=2.20.0'   'sentence-transformers>=3.0.0'   accelerate   pandas numpy scikit-learn tqdm matplotlib pyyaml

## 2. Optional: mount Google Drive

This saves embeddings and CSV results permanently. If you do not mount Drive, results will stay only in the Colab runtime.

In [ ]:
from google.colab import drive
USE_DRIVE = True

if USE_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/indic_embedding_benchmark'
else:
    BASE_DIR = '/content/indic_embedding_benchmark'

print('Saving outputs to:', BASE_DIR)

## 3. Imports and configuration

First run only 100 examples. Once the notebook works, change:

```python
QUICK_N = 250
```

Later, for full devtest:

```python
QUICK_N = 0
```


In [ ]:
import gc
import os
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

SEED = 42
SPLIT = 'devtest'
QUICK_N = 100      # start with 100; later use 250; use 0 for full split
BATCH_SIZE = 16    # safe for Colab T4; increase to 32 if memory is fine
MAX_LENGTH = 128

OUTPUT_DIR = Path(BASE_DIR) / 'outputs' / 'flores_alignment_colab'
(OUTPUT_DIR / 'embeddings').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'errors').mkdir(parents=True, exist_ok=True)

LANGS = {
    'en': 'eng_Latn',
    'hi': 'hin_Deva',
    'bn': 'ben_Beng',
    'te': 'tel_Telu',
    'ta': 'tam_Taml',
    'ml': 'mal_Mlym',
}

# Start with 3 models if you want a fast first run.
# Then add the remaining models.
MODELS = [
    {'name': 'labse', 'hf_id': 'sentence-transformers/LaBSE', 'kind': 'sentence_transformer'},
    {'name': 'mpnet_multilingual', 'hf_id': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2', 'kind': 'sentence_transformer'},
    {'name': 'muril', 'hf_id': 'google/muril-base-cased', 'kind': 'hf_mean_pool'},
    {'name': 'indicbertv2_ss', 'hf_id': 'ai4bharat/IndicBERTv2-SS', 'kind': 'hf_mean_pool', 'trust_remote_code': True},
    {'name': 'xlm_roberta_base', 'hf_id': 'FacebookAI/xlm-roberta-base', 'kind': 'hf_mean_pool'},
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Output dir:', OUTPUT_DIR)

## 4. Load FLORES parallel sentences

FLORES rows align by index across languages. So row `i` in English is the translation pair for row `i` in Hindi/Tamil/etc.

In [ ]:
def load_flores_sentences(langs: Dict[str, str], split: str, quick_n: int) -> Dict[str, List[str]]:
    all_texts = {}
    for short, flores_code in langs.items():
        print(f'Loading {short} / {flores_code}')
        ds = load_dataset('facebook/flores', flores_code, split=split, trust_remote_code=True)
        texts = [str(row['sentence']) for row in ds]
        if quick_n and quick_n > 0:
            texts = texts[:quick_n]
        all_texts[short] = texts

    lengths = {lang: len(texts) for lang, texts in all_texts.items()}
    print('Loaded lengths:', lengths)
    if len(set(lengths.values())) != 1:
        raise ValueError(f'Language lengths are not equal: {lengths}')
    return all_texts

texts_by_lang = load_flores_sentences(LANGS, SPLIT, QUICK_N)

# Show one example pair
print('
Example:')
print('EN:', texts_by_lang['en'][0])
print('HI:', texts_by_lang['hi'][0])

## 5. Embedding helpers

Sentence-transformer models use their built-in pooling. Raw backbone models use attention-mask-aware mean pooling.

In [ ]:
@dataclass
class ModelSpec:
    name: str
    hf_id: str
    kind: str
    trust_remote_code: bool = False


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


class Embedder:
    def __init__(self, spec: ModelSpec, device: str, max_length: int):
        self.spec = spec
        self.device = device
        self.max_length = max_length

        if spec.kind == 'sentence_transformer':
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(
                spec.hf_id,
                device=device,
                trust_remote_code=spec.trust_remote_code,
            )
            # Save VRAM on Colab GPU.
            if device == 'cuda':
                self.model = self.model.half()
            self.tokenizer = None

        elif spec.kind == 'hf_mean_pool':
            from transformers import AutoModel, AutoTokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(
                spec.hf_id,
                trust_remote_code=spec.trust_remote_code,
            )
            dtype = torch.float16 if device == 'cuda' else torch.float32
            self.model = AutoModel.from_pretrained(
                spec.hf_id,
                trust_remote_code=spec.trust_remote_code,
                torch_dtype=dtype,
            )
            self.model.to(device)
            self.model.eval()
        else:
            raise ValueError(f'Unknown model kind: {spec.kind}')

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int) -> np.ndarray:
        if self.spec.kind == 'sentence_transformer':
            emb = self.model.encode(
                texts,
                batch_size=batch_size,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
            return emb.astype('float32')

        all_embeddings = []
        for start in tqdm(range(0, len(texts), batch_size), desc=f'Encoding {self.spec.name}'):
            batch = texts[start:start + batch_size]
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}
            outputs = self.model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded['attention_mask'])
            pooled = F.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.detach().cpu().float().numpy())

        return np.vstack(all_embeddings).astype('float32')


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 6. Metric functions

In [ ]:
def compute_retrieval_metrics(src_emb: np.ndarray, tgt_emb: np.ndarray, top_ks: Tuple[int, ...] = (1, 5, 10)) -> Dict[str, float]:
    sim = np.matmul(src_emb, tgt_emb.T)
    n = sim.shape[0]
    sorted_idx = np.argsort(-sim, axis=1)
    ranks = np.empty(n, dtype=np.int64)

    for i in range(n):
        ranks[i] = int(np.where(sorted_idx[i] == i)[0][0]) + 1

    out = {
        'mrr': float(np.mean(1.0 / ranks)),
        'mean_rank': float(np.mean(ranks)),
        'median_rank': float(np.median(ranks)),
    }
    for k in top_ks:
        out[f'recall_at_{k}'] = float(np.mean(ranks <= k))
    out['accuracy_at_1'] = out['recall_at_1']
    return out


def compute_cosine_gap(src_emb: np.ndarray, tgt_emb: np.ndarray, seed: int = 42) -> Dict[str, float]:
    rng = np.random.default_rng(seed)
    n = src_emb.shape[0]
    pos = np.sum(src_emb * tgt_emb, axis=1)

    neg_idx = rng.permutation(n)
    for i in range(n):
        if neg_idx[i] == i:
            neg_idx[i] = (neg_idx[i] + 1) % n
    neg = np.sum(src_emb * tgt_emb[neg_idx], axis=1)

    return {
        'positive_cosine_mean': float(np.mean(pos)),
        'positive_cosine_std': float(np.std(pos)),
        'random_cosine_mean': float(np.mean(neg)),
        'random_cosine_std': float(np.std(neg)),
        'cosine_gap': float(np.mean(pos) - np.mean(neg)),
    }


def collect_errors(src_texts, tgt_texts, src_emb, tgt_emb, n_examples=25):
    sim = np.matmul(src_emb, tgt_emb.T)
    pred_idx = np.argmax(sim, axis=1)
    rows = []
    for i, p in enumerate(pred_idx):
        if p != i:
            rows.append({
                'row_id': i,
                'source_text': src_texts[i],
                'gold_translation': tgt_texts[i],
                'predicted_neighbor': tgt_texts[p],
                'gold_cosine': float(sim[i, i]),
                'predicted_cosine': float(sim[i, p]),
                'margin_pred_minus_gold': float(sim[i, p] - sim[i, i]),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.sort_values('margin_pred_minus_gold', ascending=False).head(n_examples)

## 7. Run benchmark

This cell saves embeddings after every model/language. If Colab disconnects, rerun the notebook and it will reuse cached `.npy` files.

In [ ]:
all_rows = []

for model_dict in MODELS:
    spec = ModelSpec(**model_dict)
    print(f"
===== Model: {spec.name} | {spec.hf_id} =====")

    try:
        embedder = Embedder(spec, DEVICE, MAX_LENGTH)
        emb_by_lang = {}

        for lang, texts in texts_by_lang.items():
            cache_path = OUTPUT_DIR / 'embeddings' / f"{spec.name}_{lang}_{SPLIT}_n{QUICK_N or 'full'}_l{MAX_LENGTH}.npy"
            if cache_path.exists():
                print('Loading cached:', cache_path.name)
                emb = np.load(cache_path)
            else:
                emb = embedder.encode(texts, batch_size=BATCH_SIZE)
                np.save(cache_path, emb)
                print('Saved:', cache_path.name)
            emb_by_lang[lang] = emb

        for tgt_lang in [lang for lang in LANGS if lang != 'en']:
            src_lang = 'en'
            pair = f'{src_lang}-{tgt_lang}'
            src_emb = emb_by_lang[src_lang]
            tgt_emb = emb_by_lang[tgt_lang]

            gap = compute_cosine_gap(src_emb, tgt_emb, SEED)
            retrieval = compute_retrieval_metrics(src_emb, tgt_emb)

            row = {
                'model': spec.name,
                'hf_id': spec.hf_id,
                'language_pair': pair,
                'split': SPLIT,
                'n_examples': len(texts_by_lang[src_lang]),
                **gap,
                **retrieval,
            }
            all_rows.append(row)
            print(pair, 'Acc@1:', round(row['accuracy_at_1'], 4), 'Cosine gap:', round(row['cosine_gap'], 4))

            errors = collect_errors(
                texts_by_lang[src_lang],
                texts_by_lang[tgt_lang],
                src_emb,
                tgt_emb,
                n_examples=25,
            )
            errors.to_csv(OUTPUT_DIR / 'errors' / f'{spec.name}_{pair}_errors.csv', index=False)

    except RuntimeError as e:
        print('RuntimeError for model:', spec.name)
        print(e)
        print('Tip: reduce BATCH_SIZE to 8 and rerun this cell.')
    finally:
        try:
            del embedder
        except Exception:
            pass
        clear_memory()

metrics_df = pd.DataFrame(all_rows)
metrics_path = OUTPUT_DIR / 'flores_alignment_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print('
Saved metrics:', metrics_path)
metrics_df.head()

## 8. Model summary table

In [ ]:
summary = (
    metrics_df
    .groupby('model')[['accuracy_at_1', 'recall_at_10', 'mrr', 'cosine_gap']]
    .mean()
    .sort_values('accuracy_at_1', ascending=False)
)
summary_path = OUTPUT_DIR / 'model_summary.csv'
summary.to_csv(summary_path)
print('Saved summary:', summary_path)
summary

## 9. Plot Accuracy@1

In [ ]:
pivot = metrics_df.pivot(index='model', columns='language_pair', values='accuracy_at_1')
ax = pivot.plot(kind='bar', figsize=(12, 5))
ax.set_ylabel('Accuracy@1')
ax.set_title('FLORES translation retrieval: English → Indic')
ax.legend(title='Language pair', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plot_path = OUTPUT_DIR / 'accuracy_at_1_by_model.png'
plt.savefig(plot_path, dpi=200)
plt.show()
print('Saved plot:', plot_path)

## 10. Inspect error examples

This is useful for your report. It shows where the model chose the wrong nearest translation.

In [ ]:
import glob
error_files = sorted(glob.glob(str(OUTPUT_DIR / 'errors' / '*_errors.csv')))
print('Number of error files:', len(error_files))
print('
'.join(error_files[:5]))

# Change this index to inspect another model/language pair.
if error_files:
    sample_errors = pd.read_csv(error_files[0])
    display(sample_errors.head(10))

## 11. How to interpret results

Use this simple interpretation:

- **High positive cosine mean**: translation pairs are close.
- **Low random cosine mean**: unrelated cross-language pairs are not too close.
- **Large cosine gap**: good alignment separation.
- **High Accuracy@1**: the model can retrieve the correct translation as the nearest neighbor.
- **High Recall@10**: the correct translation is at least among the top 10 candidates.

For the mentor report, prioritize models with strong **Accuracy@1 + MRR + cosine gap**.
